# Adding More Data to Prepared Labels

When we prepared the original data labels, I only included if the tile was a positive or negative example. If we want to do more detailed tasks, such as categorizing the storm according to the Saffir-Simpson hurricane speed scale, we need to pull more data into the data labels. There's a lot of fields in the IBTrACS data that we can pull back in, so let's go grab that and create a file with a little more of the observational data.

## Imports

Import everything we need for this notebook.

In [2]:
import json
import pandas as pd
import numpy as np

## Read and Format Data

We need to read the labels created when we first downloaded the data, as well as the IBTrACS data.

In [3]:
# Read labels from downloaded image data
original_labels_path = '/Users/dylanwhite/Projects/tropical-cv/data/training/image_data.json'
with open(original_labels_path,'r') as f:
    labels_data = json.load(f)

image_df = pd.DataFrame(labels_data['images'])
image_df.head()

,category,file_name,id,width,height,band,original_file,original_ul,track_coordinates,date,df_index
0,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,0,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[1176, 2555]","[-66.5, 17.4]",2022-09-18 15:00:00,8466
1,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,1,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[1153, 574]","[-106.9, 18.3]",2022-09-18 15:00:00,8415
2,negative,/Users/dylanwhite/Projects/tropical-cv/data/tr...,2,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[3036, 1670]","[-9999, -9999]",2022-09-18 15:00:00,-9999
3,negative,/Users/dylanwhite/Projects/tropical-cv/data/tr...,3,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[3779, 4009]","[-9999, -9999]",2022-09-18 15:00:00,-9999
4,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,4,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20213090000204_e202...,"[131, 3395]","[-39.0, 42.3]",2021-11-05 00:00:00,7595


In [4]:
# Read IBTrACS data
ibtracs_path = '/Users/dylanwhite/Projects/tropical-cv/data/ibtracs/ibtracs_goes_east.csv'
ibtracs_df = pd.read_csv(ibtracs_path)

# Format IBTrACS data to do the interpolation later
ibtracs_df.replace(' ',np.nan,inplace=True)
ibtracs_df['WMO_PRES'] = ibtracs_df['WMO_PRES'].astype(float)
ibtracs_df['WMO_WIND'] = ibtracs_df['WMO_WIND'].astype(float)
ibtracs_df.head()

,Unnamed: 0,index,SID,SEASON,NUMBER,NAME,ISO_TIME,NATURE,LAT,LON,WMO_WIND,WMO_PRES,TRACK_TYPE,DIST2LAND,LANDFALL,IFLAG,STORM_SPEED,STORM_DIR
0,0,666805,2017106N36310,2017,20,ARLENE,2017-04-16 06:00:00,ET,35.8,-50.3,55.0,992.0,main,1225,1225,O______________,10,135
1,1,666806,2017106N36310,2017,20,ARLENE,2017-04-16 09:00:00,ET,35.5,-49.9,NaN,NaN,main,1265,1265,P______________,10,135
2,2,666807,2017106N36310,2017,20,ARLENE,2017-04-16 12:00:00,ET,35.1,-49.5,55.0,989.0,main,1316,1316,O______________,10,135
3,3,666808,2017106N36310,2017,20,ARLENE,2017-04-16 15:00:00,ET,34.7,-49.1,NaN,NaN,main,1367,1367,P______________,10,135
4,4,666809,2017106N36310,2017,20,ARLENE,2017-04-16 18:00:00,ET,34.4,-48.7,55.0,986.0,main,1408,1408,O______________,10,135


## Interpolate Values

IBTrACS data is shown every 3 hours. Observational data, like wind speed and pressure, are only provided for hours 00, 06, 12, and 18. We downloaded the imagery without concern for this, but we want to assign additional data like wind speed to the image labels which may fall on the odd-numbered hours in the dataset. To fix this, we can group by a particular storm (using the unique `sid` key) and linearly interpolate the missing values. This should be more than good enough for our purposes.

In [5]:
# Interpolate values for odd-numbered hours that were originally missing
groups = []
for sid, group in ibtracs_df.groupby('SID'):
    group[['WMO_WIND','WMO_PRES']] = group[['WMO_WIND','WMO_PRES']].interpolate(method='linear',axis=0, limit_direction='both')
    groups.append(group)
ibtracs_df = pd.concat(groups)

# Replace NAs that weren't able to be interpolated back the ' '
ibtracs_df = ibtracs_df.fillna(' ')
ibtracs_df.head()

,Unnamed: 0,index,SID,SEASON,NUMBER,NAME,ISO_TIME,NATURE,LAT,LON,WMO_WIND,WMO_PRES,TRACK_TYPE,DIST2LAND,LANDFALL,IFLAG,STORM_SPEED,STORM_DIR
0,0,666805,2017106N36310,2017,20,ARLENE,2017-04-16 06:00:00,ET,35.8,-50.3,55.0,992.0,main,1225,1225,O______________,10,135
1,1,666806,2017106N36310,2017,20,ARLENE,2017-04-16 09:00:00,ET,35.5,-49.9,55.0,990.5,main,1265,1265,P______________,10,135
2,2,666807,2017106N36310,2017,20,ARLENE,2017-04-16 12:00:00,ET,35.1,-49.5,55.0,989.0,main,1316,1316,O______________,10,135
3,3,666808,2017106N36310,2017,20,ARLENE,2017-04-16 15:00:00,ET,34.7,-49.1,55.0,987.5,main,1367,1367,P______________,10,135
4,4,666809,2017106N36310,2017,20,ARLENE,2017-04-16 18:00:00,ET,34.4,-48.7,55.0,986.0,main,1408,1408,O______________,10,135


## Select Data to Add

Now we'll go through and add more data fields. I've chosen to add wind speed, pressure, storm translation speed and direction, and the Saffir-Simpson categorization.

In [6]:
# Define TS categorization function
def saffir_simpson_category(windspeed_mph):
    if windspeed_mph == ' ':
        return ' '
    if windspeed_mph < 74:
        return "Tropical Storm or Below"
    elif windspeed_mph <= 95:
        return "Category 1"
    elif windspeed_mph <= 110:
        return "Category 2"
    elif windspeed_mph <= 129:
        return "Category 3"
    elif windspeed_mph <= 156:
        return "Category 4"
    else:
        return "Category 5"

# Extract more metadata
tc_observation_labels = []
for image in labels_data['images']:
    if image['category']=='positive':
        ibtracs_row = ibtracs_df.loc[
            (ibtracs_df['LON']==image['track_coordinates'][0]) &
            (ibtracs_df['LAT']==image['track_coordinates'][1]) &
            (ibtracs_df['ISO_TIME']==image['date'])
        ]
        tc_observation_labels.append({
            'image_id':image['id'],
            'nature':ibtracs_row['NATURE'].item(),
            'wind_speed':ibtracs_row['WMO_WIND'].item(),
            'pressure':ibtracs_row['WMO_PRES'].item(),
            'storm_speed':ibtracs_row['STORM_SPEED'].item(),
            'storm_dir':ibtracs_row['STORM_DIR'].item(),
            'storm_category':saffir_simpson_category(ibtracs_row['WMO_WIND'].item())
        })

tc_observations_df = pd.DataFrame(tc_observation_labels)
tc_observations_df.loc[tc_observations_df['wind_speed']==' ',['wind_speed','pressure','storm_category']]=-9999
tc_observations_df.tail()

,image_id,nature,wind_speed,pressure,storm_speed,storm_dir,storm_category
490,979,TS,30.0,1007.0,9,285,Tropical Storm or Below
491,981,TS,52.5,997.0,6,290,Tropical Storm or Below
492,983,TS,85.0,972.0,7,285,Category 1
493,985,TS,107.5,958.0,9,295,Category 2
494,987,TS,35.0,1008.0,18,280,Tropical Storm or Below


## Join to Original Data

Now we just need to join the new data to our original data.

In [7]:
new_image_df = pd.merge(image_df,tc_observations_df,how='left',left_on='id',right_on='image_id')
new_image_df = new_image_df.infer_objects(copy=False).fillna(-9999)
new_image_df = new_image_df.astype({
    'image_id':int,
    'wind_speed':float,
    'pressure':float,
    'storm_speed':float,
    'storm_dir':float
})
new_image_df.head()

,category,file_name,id,width,height,band,original_file,original_ul,track_coordinates,date,df_index,image_id,nature,wind_speed,pressure,storm_speed,storm_dir,storm_category
0,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,0,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[1176, 2555]","[-66.5, 17.4]",2022-09-18 15:00:00,8466,0,TS,70.0,988.0,9.0,310.0,Tropical Storm or Below
1,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,1,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[1153, 574]","[-106.9, 18.3]",2022-09-18 15:00:00,8415,1,TS,40.0,997.0,7.0,335.0,Tropical Storm or Below
2,negative,/Users/dylanwhite/Projects/tropical-cv/data/tr...,2,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[3036, 1670]","[-9999, -9999]",2022-09-18 15:00:00,-9999,-9999,-9999,-9999.0,-9999.0,-9999.0,-9999.0,-9999
3,negative,/Users/dylanwhite/Projects/tropical-cv/data/tr...,3,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20222611500205_e202...,"[3779, 4009]","[-9999, -9999]",2022-09-18 15:00:00,-9999,-9999,-9999,-9999.0,-9999.0,-9999.0,-9999.0,-9999
4,positive,/Users/dylanwhite/Projects/tropical-cv/data/tr...,4,1200,1200,13,OR_ABI-L1b-RadF-M6C13_G16_s20213090000204_e202...,"[131, 3395]","[-39.0, 42.3]",2021-11-05 00:00:00,7595,4,TS,45.0,992.0,5.0,90.0,Tropical Storm or Below


## Write Data to File

Finally, we'll write this new data to a file.

In [10]:
new_image_df['storm_category'].unique()

array(['Tropical Storm or Below', -9999, 'Category 1', 'Category 3',
       'Category 2', 'Category 4'], dtype=object)

In [9]:
new_data = {'images':new_image_df.to_dict(orient='records')}
new_labels_path = '/Users/dylanwhite/Projects/tropical-cv/data/training/image_data_with_obs_data.json'
with open(new_labels_path,'w') as f:
    json.dump(new_data,f)